In [ ]:
import numpy as np
import pandas as pd

# DONE in extractors/common.py
def ensure_length_col(df: pd.DataFrame, text_col="review_text", length_col="len"):
    """
    Adds a column df[length_col] = length of each sentence string.

    Why?
      - We enforce length budget L in extraction
      - We optionally penalize too-long sentences

    If already present, does nothing.
    """
    if length_col not in df.columns:
        df = df.copy()
        df[length_col] = df[text_col].fillna("").astype(str).str.len().astype(int)
    return df

# DONE in extractors/common.py
def get_phi_matrix(df: pd.DataFrame, phi_cols):
    """
    Extract aspect-score columns from df and return a numpy matrix Phi.

    Phi shape: (N, K)
      - N = number of rows (sentences/reviews)
      - K = number of aspects

    We also row-normalize so each row sums to 1 (simplex).
    """
    Phi = df[phi_cols].to_numpy(dtype=np.float32)
    # (optional) row-normalize if you want exact simplex rows # phi_i/sum(phi_i)
    row_sums = Phi.sum(axis=1, keepdims=True)
    Phi = Phi / (row_sums + 1e-12)
    return Phi

In [ ]:
# --- Similarity utilities ---
# DONE in extractors/mmr.py
def cosine_sim_matrix(S: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """
    Compute cosine similarity matrix for sentence vectors S.

    S: (n, d)
    returns Sim: (n, n) with Sim[i,j] = cosine(s_i, s_j)
    """
    S = np.asarray(S, dtype=np.float32)
    norms = np.linalg.norm(S, axis=1, keepdims=True)
    Sn = S / (norms + eps)
    return Sn @ Sn.T

#DONE in extractors/mmr.py
def build_sentence_vectors_tfidf(texts, max_features: int = 5000):
    """
    Lightweight baseline for sentence vectors s_i using TF-IDF.
    Produces S: (n, d).

    # help me understand this function:
        - We use sklearn's TfidfVectorizer to convert raw text sentences into TF-IDF vectors.
        - max_features limits the vocabulary size to the top max_features terms.
        - stop_words="english" removes common English words that are not informative.
        - We handle None values by converting them to empty strings.
        - Finally, we convert the sparse matrix output to a dense numpy array of type float32.
        - The resulting S matrix can be used for cosine similarity or other downstream tasks.
        - This is a simple baseline; more sophisticated methods could use pretrained embeddings or fine-tuning.
    """
    from sklearn.feature_extraction.text import TfidfVectorizer
    texts = ["" if t is None else str(t) for t in texts]
    vec = TfidfVectorizer(max_features=max_features, stop_words="english")
    X = vec.fit_transform(texts)
    return X.toarray().astype(np.float32)

# what is TF-IDF?
# TF-IDF stands for Term Frequency-Inverse Document Frequency. 
# It is a numerical statistic that reflects how important a word is to a document in a collection or corpus. 
# The TF-IDF value increases proportionally to the number of times a word appears in the document (term frequency) 
# but is offset by the number of documents in the corpus that contain the word (inverse document frequency), 
# which helps to adjust for the fact that some words appear more frequently in general. 
# TF-IDF is commonly used in information retrieval and text mining to evaluate the relevance of a word to a document.

In [3]:
# this is section 3.1
def compute_sentence_utilities(df_candidates, phi_cols, w_u, lam=0.0, len_col="len"):
    """
    For a product p, we have candidate sentences.

    Compute for each sentence i:
      base_i = w_u^T phi_i   (how much it matches the user's aspects)

    Optionally penalize long sentences:
      base_i -= lam * len_i / mean_len

    This is the relevance score U_i for each candidate sentence.
    """
    Phi = get_phi_matrix(df_candidates, phi_cols)         # (n_p, K)
    w_u = np.asarray(w_u, dtype=np.float32)               # (K,)

    # aspect relevance score per sentence
    U = Phi @ w_u                                         # (n_p,)

    # optional length penalty
    if lam > 0:
        lens = df_candidates[len_col].to_numpy(dtype=np.float32)
        mean_len = float(np.mean(lens) + 1e-12)
        U = U - lam * (lens / mean_len)

    return U, Phi

In [4]:
# --- Deterministic MMR greedy (NEW) ---

def mmr_select_indices(
    Rel: np.ndarray,          # (n,) relevance scores U_i = w_u^T phi_i
    Sim: np.ndarray,          # (n,n)  similarity matrix : cosine on TF-IDF sentence vectors
    lengths: np.ndarray,      # (n,)  sentence lengths for length budget constraint
    k: int = 8,
    L: int = 800,
    lam_mmr: float = 0.7,     # λ in your writeup
):
    """
    Implements Algorithm MMR-Select from your writeup:

      Δ(j | S) = λ Rel[j] - (1-λ) max_{i in S} Sim[i,j]

    with feasibility constraints:
      |S| <= k,  sum(lengths[i]) <= L

    Returns selected indices (local to candidate set).
    """
    n = len(Rel) # number of candidate sentences for this product
    Rel = np.asarray(Rel, dtype=np.float32) 
    Sim = np.asarray(Sim, dtype=np.float32)
    lengths = np.asarray(lengths, dtype=np.int32)

    selected = []
    used_len = 0

    # maintain redundancy term efficiently:
    # max_sim_to_S[j] = max_{i in selected} Sim[i, j]
    max_sim_to_S = np.zeros(n, dtype=np.float32)

    for t in range(k): # try to pick up to k sentences
        best_j = None
        best_gain = -1e18 

        for j in range(n): # loop over candidates to find best one to add
            if j in selected:
                continue
            if used_len + int(lengths[j]) > L:
                continue

            redundancy = 0.0 if len(selected) == 0 else float(max_sim_to_S[j])

            gain = lam_mmr * float(Rel[j]) - (1.0 - lam_mmr) * redundancy

            if gain > best_gain:
                best_gain = gain
                best_j = j

        if best_j is None:
            break

        selected.append(best_j)
        used_len += int(lengths[best_j])

        # update max similarity to selected set for all j in O(n)
        max_sim_to_S = np.maximum(max_sim_to_S, Sim[best_j])

    return np.array(selected, dtype=int), used_len


In [5]:
def extract_mmr_for_user_product(
    df_sent: pd.DataFrame,
    product_id: str,
    phi_cols,
    w_u: np.ndarray,
    k: int = 8,
    L: int = 800,
    lam_mmr: float = 0.7,
    text_col: str = "review_text",
    len_col: str = "len",
    sim_method: str = "tfidf",
    tfidf_max_features: int = 5000,
    length_penalty: float = 0.0,   # optional: if you want soft penalty in Rel
):
    """
    Deterministic MMR extraction for one product:

    1) build candidate set for product
    2) compute Rel[j] = w^T phi_j   (reuses compute_sentence_utilities)
    3) build sentence vectors s_j (TF-IDF baseline)
    4) compute cosine Sim
    5) run greedy MMR selection under (k, L)
    """
    # REUSE: add length column
    df_sent = ensure_length_col(df_sent, text_col=text_col, length_col=len_col)

    cand = df_sent[df_sent["parent_asin"] == product_id].copy()
    if len(cand) == 0:
        return None

    # REUSE: compute relevance utilities (Rel) and Phi
    Rel, Phi = compute_sentence_utilities(
        cand, phi_cols, w_u, lam=length_penalty, len_col=len_col
    )  # Rel: (n,), Phi: (n,K)

    # NEW: build sentence vectors and similarity
    if sim_method == "tfidf":
        S = build_sentence_vectors_tfidf(cand[text_col].tolist(), max_features=tfidf_max_features)
    else:
        raise ValueError("Only sim_method='tfidf' implemented in this baseline file.")

    Sim = cosine_sim_matrix(S)

    lengths = cand[len_col].to_numpy(dtype=np.int32)

    sel_idx, used_len = mmr_select_indices(
        Rel=Rel, Sim=Sim, lengths=lengths, k=k, L=L, lam_mmr=lam_mmr
    )

    selected_df = cand.iloc[sel_idx].copy()
    selected_df["Rel"] = Rel[sel_idx]

    # redundancy diagnostics: max sim to earlier selected
    # compute in selected order
    red = []
    for t, j in enumerate(sel_idx):
        if t == 0:
            red.append(0.0)
        else:
            prev = sel_idx[:t]
            red.append(float(np.max(Sim[prev, j])))
    selected_df["max_sim_to_prev"] = red

    z_t = Phi[sel_idx].mean(axis=0) if len(sel_idx) > 0 else None

    return {
        "selected_df": selected_df,   # keep order as selected (greedy order)
        "selected_idx": sel_idx,
        "z_t": z_t,
        "used_len": used_len,
        "n_candidates": len(cand),
    }

In [6]:
data = pd.read_csv("../data/train_with_aspect_scores.csv")
data.head()

,user_id,parent_asin,rating,timestamp,history,review_title,review_text,helpful_vote,verified_purchase,main_category,...,asp_12,asp_13,asp_14,asp_15,asp_16,asp_17,asp_18,asp_19,top_aspect,top_aspect_score
0,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B081TJ8YS3,4.0,1588615855,NaN,Works great but smells a little weird.,"This product does what I need it to do, I just...",1,True,All Beauty,...,0.000672,0.000268,0.017212,0.004381,0.000119,0.739653,0.000900,0.000672,asp_17,0.739653
1,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B00YQ6X8EO,5.0,1588687728,B081TJ8YS3,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,0,True,All Beauty,...,0.000535,0.000037,0.000658,0.001067,0.000023,0.071295,0.000102,0.000051,asp_7,0.860609
2,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,B097R46CSY,5.0,1589665266,NaN,Yes!,"Smells good, feels great!",2,True,All Beauty,...,0.002384,0.005362,0.001521,0.000309,0.000172,0.959386,0.000226,0.002328,asp_17,0.959386
3,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,B08BZ63GMJ,5.0,1609322563,NaN,A+,Love it,0,True,All Beauty,...,0.005844,0.034661,0.000687,0.000209,0.000027,0.000094,0.000344,0.002957,asp_9,0.897146
4,AGMJ3EMDVL6OWBJF7CA5RGJLXN5A,B00R8DXL44,4.0,1598567408,NaN,Pretty Color,The polish was quiet thick and did not apply s...,0,True,All Beauty,...,0.014724,0.002269,0.016684,0.269764,0.013187,0.001529,0.002464,0.003324,asp_4,0.480720


In [7]:
data["parent_asin"].value_counts()

parent_asin
B085BB7B1M    1828
B0BM4GX6TT    1609
B07C533XCW    1310
B00R1TAN7I    1309
B019GBG0IE    1197
              ... 
B07PGQ5JH5       1
B085LK54NM       1
B01AO7EXI8       1
B07S1S658F       1
B07ZPVM149       1
Name: count, Length: 95431, dtype: int64

In [8]:
K = 20
phi_cols = [f"asp_{k}" for k in range(K)]
phi_cols

['asp_0',
 'asp_1',
 'asp_2',
 'asp_3',
 'asp_4',
 'asp_5',
 'asp_6',
 'asp_7',
 'asp_8',
 'asp_9',
 'asp_10',
 'asp_11',
 'asp_12',
 'asp_13',
 'asp_14',
 'asp_15',
 'asp_16',
 'asp_17',
 'asp_18',
 'asp_19']

In [9]:
def softmax(x: np.ndarray, axis=-1):
    """
    softmax turns arbitrary real numbers into a probability distribution. 

    # we would need this to convert aspect scores phi into proper preference vector probabilities.

    Example:
        x = [2, 1, 0]
        softmax(x) -> [0.66, 0.24, 0.09] (roughly)

    Why subtract max?
        Prevents exp(large number) overflow.
    """
    x = np.asarray(x)
    x = x - np.max(x, axis=axis, keepdims=True)  # numerical stability
    ex = np.exp(x)
    return ex / (np.sum(ex, axis=axis, keepdims=True) + 1e-12)

# section 3.3 uniform averaging without feedback weighting
def estimate_user_pref_from_own_text(df_sent, user_id, phi_cols, beta=5.0):
    """
    Bootstraps w_u directly from the user's own sentences/reviews  if present 
    """
    sub = df_sent[df_sent["user_id"] == user_id]
    if len(sub) == 0:
        K = len(phi_cols)
        return np.ones(K, dtype=np.float32) / K
    Phi = get_phi_matrix(sub, phi_cols)      # (m,K)
    ztilde = Phi.mean(axis=0)                # average aspect mass
    return softmax(beta * ztilde, axis=0)

In [10]:
w_u = estimate_user_pref_from_own_text(data, user_id="AG73BVBKUOH22USSFJA5ZWL7AKXA", phi_cols=phi_cols, beta=8.0)

out_mmr = extract_mmr_for_user_product(
    df_sent=data,
    product_id="B085BB7B1M",
    phi_cols=phi_cols,
    w_u=w_u,
    k=10,
    L=900,
    lam_mmr=0.7,
    sim_method="tfidf",
)

selected_df = out_mmr["selected_df"]
z_t = out_mmr["z_t"]
print("Aspect profile z_t:", z_t)

Aspect profile z_t: [1.1506974e-03 6.4749449e-02 1.3185926e-01 7.0021456e-01 4.3830290e-03
 9.8028884e-04 8.2762324e-04 1.3325970e-02 5.5377344e-03 6.7981794e-03
 1.3894804e-03 3.7007875e-04 2.6092348e-03 6.2156869e-03 2.7302306e-02
 1.2362572e-02 2.8244338e-03 2.9395912e-03 8.5397866e-03 5.6200405e-03]


In [11]:
selected_df[["review_text", "Rel", "max_sim_to_prev", "len"]]

,review_text,Rel,max_sim_to_prev,len
252239,My Dermotologist recommend this product and I ...,0.276816,0.000000,131
261728,This is so great for body! Do NOT use on face!,0.274299,0.000000,46
393949,So far I like the product. It exfoliates we’ll...,0.274634,0.047777,132
6217,Buy it. Right now.It gives me such soft skin.,0.261735,0.033060,45
94564,"Amazing product, does what it claims and remov...",0.254586,0.034942,139
393726,I missed using this type product. Brought bac...,0.265052,0.065906,186
200811,If you want something similar to this product ...,0.270888,0.088777,140
39519,Effective on skin and good quality- I get repl...,0.246548,0.040577,72
393814,Too ruff!,0.032543,0.000000,9
157206,NaN,0.028587,0.000000,0


In [12]:
selected_df.to_csv("../data/extracted_sentences_deterministic_MMR_Style.csv", index=False)